# Lilly v2 — train the reader (OCR on Kaggle)

Scope: `docs/V2-BOUNDARIES.md`. Latin Bosnian only (č ć đ š ž). No Cyrillic.

Set these in the panel on the right:

- **Session options → Accelerator → GPU T4**
- **Session options → Internet → On**

Nothing to upload — clones GitHub, fetches base weights from Hugging Face,
generates ~20k synthetic crops, continues from **pass-1 `lilly.pth`**, trains
**7 more epochs** (10 total), ships `lilly-read.zip`.

Attach dataset **`lilly-read-pass1`** (uploaded by `kaggle_train.py ocr`).

Real hand-labelled crop **images** are not in git (only labels are). This run is
synthetic-heavy unless you attach a dataset.

Setup mistakes stop the run. A training run that collapses does **not** stop it:
the weights are kept for inspection and `lilly-read.zip` is simply not written,
so the log can be read instead of guessed at.

In [ ]:
# 1. Stop here unless the machine is actually set up
import os, subprocess, sys, urllib.error, urllib.request
from pathlib import Path

import torch
assert torch.cuda.is_available(), (
    "No GPU. Right panel -> Session options -> Accelerator -> GPU, then Save & Run All again.")

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(torch.cuda.device_count(), "GPU(s) visible, using:", torch.cuda.get_device_name(0))

def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=20).close()
    except urllib.error.HTTPError:
        pass
    except Exception as exc:
        raise SystemExit(
            f"Cannot reach {url} ({exc}). Right panel -> Session options -> Internet -> On.")

for host in ("https://github.com", "https://pypi.org", "https://huggingface.co"):
    reachable(host)
print("network ok")

def run(*cmd):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    subprocess.run([str(c) for c in cmd], check=True)

In [ ]:
# 2. Get the Lilly code — into scratch, NOT /kaggle/working
# Everything under /kaggle/working becomes Output. Training copies ~18,500
# synthetic PNGs into data/ocr/train and valid, and with the clone there too,
# `kaggle kernels output` spent 172 s on PNGs and git objects and never reached
# the weights zip at all. Only the zips below belong in Output.
SCRATCH = Path("/kaggle/temp") if Path("/kaggle/temp").is_dir() else Path("/tmp")
CLONE = SCRATCH / "Lilly"
subprocess.run(["rm", "-rf", str(CLONE)], check=True)
os.chdir(SCRATCH)
run("git", "clone", "-q", "https://github.com/ssaaffaakk/Lilly.git")
assert (CLONE / "training" / "train_ocr.py").is_file(), "clone produced nothing"
os.chdir(CLONE)
print("working in", os.getcwd(), "— Output will hold only the zips")

In [ ]:
# 3. Install what we need (~2 min)
# Do NOT pip-install torch from requirements.txt — that file pins CPU wheels for
# the Mac. Kaggle already ships a GPU build; replacing it wastes time and can
# break CUDA.
NEEDED = ["easyocr", "opencv-python-headless", "pillow", "huggingface_hub"]
pins = {}
for line in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if line.startswith("--"):
        continue
    if "==" in line:
        pins[line.split("==")[0].strip().lower()] = line
run(sys.executable, "-m", "pip", "install", "-q", *[pins.get(n, n) for n in NEEDED])

In [ ]:
# 3b. Prove the GPU can backprop before an hour is spent generating images
x = torch.randn(256, 256, device="cuda", requires_grad=True)
y = (x @ torch.randn(256, 256, device="cuda")).sum()
y.backward()
print(f"GPU backprop ok on {torch.cuda.get_device_name(0)}")
torch.cuda.empty_cache()

In [ ]:
# 4. Base reader weights from Hugging Face + pass-1 from the attached dataset
import shutil
run("python3", "scripts/fetch_models.py")
READ = Path("models/lilly/read")
base = READ / "latin_g2.pth"
assert base.is_file() and base.stat().st_size > 1_000_000, "fetch_models missing read weights"
net = READ / "user_network"
for needed in ("lilly.yaml", "lilly.py"):
    assert (net / needed).is_file(), f"fetch_models missing user_network/{needed}"

INIT = base
input_root = Path("/kaggle/input")
for ds in sorted(input_root.iterdir()) if input_root.is_dir() else []:
    candidate = ds / "read" / "lilly.pth"
    if candidate.is_file():
        shutil.copy(candidate, READ / "lilly.pth")
        un = ds / "read" / "user_network"
        if un.is_dir():
            shutil.copytree(un, net, dirs_exist_ok=True)
        INIT = READ / "lilly.pth"
        print(f"pass-1 reader from {ds.name} -> {INIT}")
        break
else:
    raise SystemExit(
        "No pass-1 weights. Launch with: python3 scripts/kaggle_train.py ocr\n"
        "(uploads lilly-read-pass1 dataset from your local lilly.pth)")

assert INIT.is_file() and INIT.stat().st_size > 100_000
print("heavy pass-2 will continue from", INIT)

In [ ]:
# 5. Synthetic Bosnian crops (~2 min)
# The clone is already on scratch, so the 20k crops here and the ~18,500 copies
# prepare_ocr_data.py makes into train/valid never touch Output.
#
# Fonts: repo has data/fonts/README only. generate_ocr_data.py falls back to
# /usr/share/fonts on Linux and exits with a clear error if none have č/đ.
run("python3", "data/scripts/generate_ocr_data.py", "--count", "20000", "--build-splits")
train_n = sum(1 for _ in open("data/ocr/train/gt.txt", encoding="utf-8"))
valid_n = sum(1 for _ in open("data/ocr/valid/gt.txt", encoding="utf-8"))
assert train_n > 5000, f"only {train_n} train crops"
assert valid_n > 100, f"only {valid_n} valid crops"
print(f"synthetic splits: train {train_n:,}, valid {valid_n:,}")

In [ ]:
# 5b. Real hand-labelled crops — only if the PNGs came with the clone
for labels in (Path("data/ocr/crops2/labels-human.tsv"),
               Path("data/ocr/crops/labels-human.tsv")):
    if not labels.is_file():
        continue
    rows = labels.read_text(encoding="utf-8").splitlines()
    if not rows:
        continue
    img_name = rows[0].split("\t")[0]
    img_path = labels.parent / img_name
    if img_path.is_file():
        run("python3", "training/prepare_ocr_data.py", "--labels", str(labels))
        print(f"merged real crops from {labels}")
        break
else:
    print("no real crop images in clone — synthetic only (expected on Kaggle)")

In [ ]:
# 5c. Four real GPU training steps before the long run (~30 s)
# Caught the cuda/cpu CTCLoss bug that killed v1 at step 1 after 2 min of setup.
print("$ python3 training/train_ocr.py --quick-test", flush=True)
smoke = subprocess.run(["python3", "training/train_ocr.py", "--quick-test"], check=False)
assert smoke.returncode == 0, (
    f"GPU OCR training smoke failed (exit {smoke.returncode}) — fix before the long run")
print("GPU OCR training smoke ok")

In [ ]:
# 6. HEAVY TRAINING — 7 epochs on pass-1 (~15 min on T4 at 0.06 s/step)
# Pass-1 was 3 epochs from latin_g2; this pass reaches 10 total, like the v2 plan.
# --keep-trained writes the weights whatever the gate then decides, so a refused
# or collapsed run still leaves something to look at. Exit 1 is one of those
# verdicts, not a crash: check the file, not the exit code.
#
# 1e-4 on CUDA grew logits 56 -> 155 by step 1257 and the loss went inf.
# 1e-5 is the rate speech already uses on this card.
os.environ["LILLY_RUN_ID"] = "heavy-pass2"
TRAINED = Path("models/lilly/read-trained.pth")
cmd = ["python3", "training/train_ocr.py", "--epochs", "7", "--batch-size", "16",
       "--lr", "1e-5", "--weights", str(INIT), "--keep-trained", str(TRAINED)]
print("$", " ".join(cmd), flush=True)
proc = subprocess.run(cmd, check=False)
assert TRAINED.is_file() and TRAINED.stat().st_size > 100_000, (
    f"training wrote no weights at all (exit {proc.returncode})")

run("zip", "-j", "/kaggle/working/lilly-read-trained.zip", str(TRAINED))
print(f"train_ocr exit {proc.returncode} — "
      f"lilly-read-trained.zip saved before any packaging")

In [ ]:
# 7. Package what the app loads — only when the install gate passed
READ = Path("models/lilly/read")
NET = READ / "user_network"
for needed in ("lilly.yaml", "lilly.py"):
    assert (NET / needed).is_file(), f"missing user_network/{needed}"

if proc.returncode != 0:
    print("=" * 70)
    print("NOT SHIPPABLE — no lilly-read.zip written.")
    print("The gate refused this reader or the run collapsed. read-trained.pth is")
    print("in Output for inspection only. Do NOT install it.")
    print("=" * 70)
else:
    assert (READ / "lilly.pth").is_file(), "gate passed but lilly.pth missing"
    run("zip", "-qr", "/kaggle/working/lilly-read.zip",
        "models/lilly/read/lilly.pth",
        "models/lilly/read/user_network/lilly.yaml",
        "models/lilly/read/user_network/lilly.py")
    size = Path("/kaggle/working/lilly-read.zip").stat().st_size
    assert size > 100_000, f"zip too small: {size}"
    print(f"lilly-read.zip — {size / 1048576:.1f} MB")

# What Output actually holds. A long list here means something heavy was written
# to /kaggle/working and the download will not reach the weights.
out = sorted(Path("/kaggle/working").rglob("*"))
print(f"\nOutput holds {len(out)} entries:")
for p in out[:20]:
    print(f"  {p.relative_to('/kaggle/working')}  {p.stat().st_size / 1048576:.1f} MB")
assert len(out) < 50, f"Output has {len(out)} entries — the fetch will drown"

**If `lilly-read.zip` is in Output:** the gate passed. Unzip it over the repo root —
it carries `models/lilly/read/lilly.pth` and `models/lilly/read/user_network/`.
Keep `latin_g2.pth` and any previous `lilly.pth` as backup.

**If only `lilly-read-trained.zip` is there:** the gate refused or the run collapsed.
Do not install it. Read the `after:` line and the `DBGLOG` lines in the log first.